In [1]:
# ============================================================
# CELL 0: Fresh notebook setup for CNN-1D / EEGNet training
# ============================================================
import random
import sys
import numpy as np
import torch
import pandas as pd
from pathlib import Path

def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

project_root = Path("/lp-dev/23BRS1328/erp_trial_min")
sys.path.insert(0, str(project_root))

print(f"project_root = {project_root}, exists = {project_root.exists()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Confirm the corrected data loader exists and works
import dl_prep.explicit_epoch_loader as eel
print(f"\nCONDITION_SPEC entries: {len(eel.CONDITION_SPEC)}")
print(f"Paradigms covered: {list(eel.CONDITION_SPEC.keys())}")

/home/nvidia/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


project_root = /lp-dev/23BRS1328/erp_trial_min, exists = True
CUDA available: True
Device: NVIDIA A100-SXM4-80GB

CONDITION_SPEC entries: 22
Paradigms covered: ['ERP_CORE_P3', 'ERP_CORE_MMN', 'ERP_CORE_ERN', 'ERP_CORE_N400', 'ERP_CORE_N2pc', 'ERP_CORE_LRP', 'ERP_CORE_N170', 'DS006018_auditoryoddball', 'DS006018_visualoddball', 'DS002680_gonogo_1', 'DS002680_gonogo_2', 'DS002680_gonogo_3', 'DS002680_gonogo_4', 'DS002680_gonogo_5', 'DS002680_gonogo_6', 'DS002680_gonogo_7', 'DS002680_gonogo_8', 'DS002680_gonogo_9', 'DS002680_gonogo_10', 'DS002680_gonogo_11', 'DS002680_gonogo_12', 'DS002680_gonogo_13']


In [2]:
# ============================================================
# CELL 1: Inventory existing CNN/EEGNet code and prior results
# ============================================================
# Look for existing model definition files
candidates = list(project_root.rglob("*cnn*.py")) + list(project_root.rglob("*eegnet*.py"))
candidates = [f for f in candidates if "__pycache__" not in str(f) and "checkpoint" not in str(f)]
print("CNN/EEGNet-related Python files found:")
for f in sorted(set(candidates)):
    print(f"  {f.relative_to(project_root)}")

print()

# Check for prior training output directories (from earlier this session)
result_dirs = [d for d in project_root.iterdir() if d.is_dir() and ("dl" in d.name.lower() or "cnn" in d.name.lower() or "eegnet" in d.name.lower())]
print("Candidate result directories:")
for d in result_dirs:
    print(f"  {d.name}")

print()

# Check existing dl_models/ directory specifically (referenced earlier this session)
dl_models_dir = project_root / "dl_models"
if dl_models_dir.exists():
    print("dl_models/ contents:")
    for f in sorted(dl_models_dir.glob("*.py")):
        print(f"  {f.name}")
else:
    print("dl_models/ does not exist")

CNN/EEGNet-related Python files found:
  dl_models/cnn1d.py
  run_phase4_train_cnn.py

Candidate result directories:
  dl_prep
  dl_models
  results_dl

dl_models/ contents:
  __init__.py
  cnn1d.py
  evaluator.py
  minimization_analyzer.py
  trainer.py


In [3]:
import shutil
from datetime import datetime

# Back up (not delete) the old results directory
results_dl = project_root / "results_dl"
if results_dl.exists():
    backup_path = project_root / f"results_dl_STALE_BACKUP_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    shutil.move(str(results_dl), str(backup_path))
    print(f"[Backup] Old results_dl moved to: {backup_path}")

# Recreate clean, empty structure
results_dl.mkdir(parents=True, exist_ok=True)
print(f"[Created] Fresh empty {results_dl}")

# Quick sanity check: does cnn1d.py look like a real, sane CNN architecture?
print("\n" + "=" * 60)
print("cnn1d.py -- architecture check")
print("=" * 60)
text = (project_root / "dl_models" / "cnn1d.py").read_text()
print(text[:2500])

[Backup] Old results_dl moved to: /lp-dev/23BRS1328/erp_trial_min/results_dl_STALE_BACKUP_20260830_132536
[Created] Fresh empty /lp-dev/23BRS1328/erp_trial_min/results_dl

cnn1d.py -- architecture check
# dl_models/cnn1d.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class ERPConv1DNet(nn.Module):
    """
    1D CNN baseline for ERP/EEG trials.

    - Input: (N, C, T) where C=26–32, T=501–1025
    - Convolutions across time; channels dimension treated as feature maps per time step.
    - AdaptiveAvgPool1d(1) collapses variable T to a fixed feature vector.

    Design: modest depth, reasonably wide, with dropout for regularization.
    """

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        base_channels: int = 64,
        dropout: float = 0.3,
    ):
        super().__init__()

        # Blocks: Conv -> BN -> ReLU
        self.block1 = nn.Sequential(
            nn.Conv1d(in_channels, base_channels, kernel_size=7, strid

In [4]:
# ============================================================
# CELL 4: Check trainer.py and evaluator.py before reuse
# ============================================================
print("=" * 60)
print("trainer.py")
print("=" * 60)
print((project_root / "dl_models" / "trainer.py").read_text())

trainer.py
# dl_models/trainer.py

from typing import Dict, List

import torch
from torch import nn
from torch.utils.data import DataLoader


class LabelSmoothingCrossEntropy(nn.Module):
    """
    Standard label-smoothed cross-entropy for multi-class (including binary as 2 classes).
    """

    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = float(smoothing)

    def forward(self, logits, target):
        """
        logits: (N, C)
        target: (N,) with class indices in [0, C-1]
        """
        n_classes = logits.size(-1)
        log_probs = torch.log_softmax(logits, dim=-1)

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)

        return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))


class Trainer:
    """
    Handles training and validation with e

In [5]:
print((project_root / "dl_models" / "evaluator.py").read_text())

# dl_models/evaluator.py

from typing import Dict, Tuple

import json
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

import torch
from torch.utils.data import DataLoader
from torch import nn


def evaluate_model(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device,
    output_dir,
    use_amp: bool = False,
    amp_dtype: torch.dtype = torch.bfloat16,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray, np.ndarray]:
    """
    Evaluate model on test_loader and save metrics / ROC curve in output_dir.
    """

    os.makedirs(output_dir, exist_ok=True)

    model.eval()
    model.to(device)

    all_logits = []
    all_targets = []

    with torch.no_grad():
        for x, y, _subjects in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=

In [6]:
# ============================================================
# CELL 5: Write dl_models/eegnet.py -- standard EEGNet architecture,
# built to handle variable channel counts (26-32) and epoch lengths
# (501-1025 samples) across all 22 paradigms without hardcoded
# FC-layer dimensions.
# ============================================================
eegnet_code = '''# dl_models/eegnet.py

import torch
import torch.nn as nn


class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al., 2018), adapted for variable channel count and
    epoch length across paradigms.

    Standard EEGNet computes a fixed FC input size assuming constant
    channels/timepoints. Since this project's 22 paradigms vary in both
    (26-32 channels, 501-1025 samples), we use AdaptiveAvgPool2d before
    the classifier instead -- avoids a class of shape-mismatch bugs that
    would otherwise require per-paradigm special-casing.

    Input: (N, 1, C, T) -- single "image" channel, C EEG channels, T samples.
    """

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        F1: int = 8,
        D: int = 2,
        F2: int = 16,
        kernel_length: int = 64,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.in_channels = in_channels

        # Block 1: temporal convolution, then depthwise spatial convolution
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kernel_length),
                      padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(in_channels, 1),
                      groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        # Block 2: separable convolution
        self.separable_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16),
                      padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        # Adaptive pooling handles variable T (and, defensively, variable
        # spatial dims) instead of a hardcoded Linear input size.
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(F2, num_classes)

    def forward(self, x):
        """
        x: (N, C, T) -- reshaped internally to (N, 1, C, T) for Conv2d.
        """
        if x.dim() == 3:
            x = x.unsqueeze(1)  # (N, 1, C, T)

        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = self.global_pool(x)       # (N, F2, 1, 1)
        x = x.flatten(1)              # (N, F2)
        return self.classifier(x)
'''

eegnet_path = project_root / "dl_models" / "eegnet.py"
eegnet_path.write_text(eegnet_code)
print(f"[Created] {eegnet_path}")

# Sanity-test on TWO different shapes -- confirms it handles variation
# across paradigms correctly, not just one "lucky" shape.
import sys
sys.

SyntaxError: invalid syntax (2405491924.py, line 93)

In [7]:
# ============================================================
# CELL 5b: Sanity-test EEGNet on two different real paradigm shapes
# ============================================================
import sys
sys.path.insert(0, str(project_root))
import dl_models.eegnet as eegnet_mod
import importlib
importlib.reload(eegnet_mod)
import torch

test_shapes = [
    (4, 30, 1025),  # ERP_CORE-style: 30 channels, 1025 samples
    (4, 26, 501),   # DS006018-style: 26 channels, 501 samples
]
for n, c, t in test_shapes:
    model = eegnet_mod.EEGNet(in_channels=c, num_classes=2)
    dummy = torch.randn(n, c, t)
    out = model(dummy)
    print(f"Input {(n,c,t)} -> Output {tuple(out.shape)} (expect ({n}, 2))")

ModuleNotFoundError: No module named 'dl_models.eegnet'

In [8]:
print((project_root / "dl_models" / "eegnet.py").exists())

False


In [9]:
# ============================================================
# CELL 5 (corrected): Write dl_models/eegnet.py and test in one cell
# ============================================================
eegnet_code = '''# dl_models/eegnet.py

import torch
import torch.nn as nn


class EEGNet(nn.Module):
    """
    EEGNet (Lawhern et al., 2018), adapted for variable channel count and
    epoch length across paradigms.

    Standard EEGNet computes a fixed FC input size assuming constant
    channels/timepoints. Since this project's 22 paradigms vary in both
    (26-32 channels, 501-1025 samples), we use AdaptiveAvgPool2d before
    the classifier instead -- avoids a class of shape-mismatch bugs that
    would otherwise require per-paradigm special-casing.

    Input: (N, 1, C, T) -- single "image" channel, C EEG channels, T samples.
    """

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        F1: int = 8,
        D: int = 2,
        F2: int = 16,
        kernel_length: int = 64,
        dropout: float = 0.5,
    ):
        super().__init__()
        self.in_channels = in_channels

        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, kernel_length),
                      padding=(0, kernel_length // 2), bias=False),
            nn.BatchNorm2d(F1),
        )
        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(in_channels, 1),
                      groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
        )

        self.separable_conv = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 16),
                      padding=(0, 8), groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(inplace=True),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(F2, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return self.classifier(x)
'''

eegnet_path = project_root / "dl_models" / "eegnet.py"
eegnet_path.write_text(eegnet_code)
print(f"[Created] {eegnet_path}, exists={eegnet_path.exists()}")

import sys
sys.path.insert(0, str(project_root))
import importlib
import dl_models.eegnet as eegnet_mod
importlib.reload(eegnet_mod)
import torch

test_shapes = [(4, 30, 1025), (4, 26, 501)]
for n, c, t in test_shapes:
    model = eegnet_mod.EEGNet(in_channels=c, num_classes=2)
    dummy = torch.randn(n, c, t)
    out = model(dummy)
    print(f"Input {(n,c,t)} -> Output {tuple(out.shape)} (expect ({n}, 2))")

[Created] /lp-dev/23BRS1328/erp_trial_min/dl_models/eegnet.py, exists=True
Input (4, 30, 1025) -> Output (4, 2) (expect (4, 2))
Input (4, 26, 501) -> Output (4, 2) (expect (4, 2))


In [10]:
# ============================================================
# CELL 6: dl_models/loso_eval.py -- lightweight per-fold metrics,
# no file writes per fold (unlike evaluator.py, which is designed
# for a single train/val/test call, not thousands of LOSO folds).
# ============================================================
loso_eval_code = '''# dl_models/loso_eval.py

import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score,
)


def evaluate_fold(model, test_loader, device, use_amp=False, amp_dtype=torch.bfloat16):
    """
    Lightweight single-fold evaluation -- returns a metrics dict, no disk I/O.
    Used inside a LOSO loop (called once per subject per step per model),
    where writing a file per call would be impractical at scale.
    """
    model.eval()
    model.to(device)

    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y, _subjects in test_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            if use_amp and device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    logits = model(x)
            else:
                logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())

    logits = torch.cat(all_logits, dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy()
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=-1)

    # Degenerate-fold guard: a single-class test fold makes several
    # metrics (AUC especially) undefined -- flag rather than crash or
    # silently emit a misleading number.
    unique_targets = np.unique(targets)
    degenerate = len(unique_targets) < 2

    metrics = {
        "accuracy": float(accuracy_score(targets, preds)),
        "balanced_accuracy": float(balanced_accuracy_score(targets, preds)) if not degenerate else float("nan"),
        "f1": float(f1_score(targets, preds, average="binary", zero_division=0)),
        "auc": float(roc_auc_score(targets, probs[:, 1])) if (not degenerate and probs.shape[1] == 2) else float("nan"),
        "n_test_trials": int(len(targets)),
        "degenerate_fold": degenerate,
    }
    return metrics
'''

loso_eval_path = project_root / "dl_models" / "loso_eval.py"
loso_eval_path.write_text(loso_eval_code)
print(f"[Created] {loso_eval_path}")

[Created] /lp-dev/23BRS1328/erp_trial_min/dl_models/loso_eval.py


In [11]:
# ============================================================
# CELL 7: Check whether raw epoch-shaped data (not flattened
# features) is available for CNN-1D/EEGNet, which need (channels,
# time) input, not the 1920-dim engineered feature vector.
# ============================================================
import numpy as np

sample_dir = project_root / "stage3_results" / "ml" / "ERP_CORE__P3"
X = np.load(sample_dir / "X.npy")
print(f"stage3_results/ml/ERP_CORE__P3/X.npy shape: {X.shape}")
print("This is the FLATTENED engineered-feature matrix (band power + PLV + wPLI + entropy),")
print("NOT raw (channels, time) epochs -- confirming CNN-1D/EEGNet need a different data source.")

# Check whether explicit_epoch_loader.py (used for classical ML's raw
# labeling) can give us raw (trials, channels, time) arrays directly
import dl_prep.explicit_epoch_loader as eel
import importlib
importlib.reload(eel)
print(f"\nload_explicit signature check -- does it return raw shaped data?")
import inspect
print(inspect.signature(eel.load_explicit))

stage3_results/ml/ERP_CORE__P3/X.npy shape: (1828, 1920)
This is the FLATTENED engineered-feature matrix (band power + PLV + wPLI + entropy),
NOT raw (channels, time) epochs -- confirming CNN-1D/EEGNet need a different data source.

load_explicit signature check -- does it return raw shaped data?
(results_batch_dir: pathlib.Path, dataset_name: str, task_name: str)


In [12]:
# ============================================================
# CELL 8: Confirm load_explicit() returns raw (trials, channels, time)
# shaped data, suitable for CNN-1D/EEGNet
# ============================================================
X, y, subject_ids, ch_names, sfreq = eel.load_explicit(
    project_root / "results_batch", "ERP_CORE", "P3"
)
print(f"X shape: {X.shape}  (expect: trials, channels, time)")
print(f"y shape: {y.shape}, classes: {np.unique(y)}, balance: {np.bincount(y)}")
print(f"n unique subjects: {len(np.unique(subject_ids))}")
print(f"n channels (from ch_names): {len(ch_names)}")
print(f"sfreq: {sfreq}")

[Explicit Loader] ERP_CORE_P3: X=(1828, 30, 1025), y=(1828,), classes=[0 1], subjects=36, class_balance=[370, 1458]
X shape: (1828, 30, 1025)  (expect: trials, channels, time)
y shape: (1828,), classes: [0 1], balance: [ 370 1458]
n unique subjects: 36
n channels (from ch_names): 30
sfreq: 1024.0


In [14]:
# ============================================================
# CELL 9: Write the CNN-1D / EEGNet LOSO training orchestration
# script -- same structure as run_stage1_classical_ml.py, adapted
# for PyTorch training instead of sklearn/xgboost.
# ============================================================
script_code = '''#!/usr/bin/env python3
"""
Stage: CNN-1D / EEGNet, LOSO cross-validation
==============================================

Models:
    - CNN-1D (dl_models/cnn1d.py)
    - EEGNet (dl_models/eegnet.py)

Evaluation:
    - Leave-One-Subject-Out cross-validation (same subject ordering
      convention as classical ML, for fold-matched comparability)
    - Trial-count subsampling grid (same 10 steps as classical ML)
    - Accuracy, balanced accuracy, F1, AUC per fold

Data source: results_batch/ via dl_prep.explicit_epoch_loader.load_explicit()
             (raw, corrected, verified epoch data -- same source used for
             this project's Phase 2 bug-fix verification)

Resumable: a completed (model, paradigm, step) combination receives a
marker file, same convention as run_stage1_classical_ml.py.

Example:
    cd /lp-dev/23BRS1328/erp_trial_min
    nohup python3 dl_models/run_dl_loso.py \\
        --shard-index 0 --num-shards 1 \\
        > logs_dl_loso.log 2>&1 &
"""

from __future__ import annotations

import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

PROJECT_ROOT = Path("/lp-dev/23BRS1328/erp_trial_min")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"Project root does not exist: {PROJECT_ROOT}")
sys.path.insert(0, str(PROJECT_ROOT))

from dl_prep.explicit_epoch_loader import load_explicit, CONDITION_SPEC
from dl_models.cnn1d import ERPConv1DNet
from dl_models.eegnet import EEGNet
from dl_models.trainer import Trainer, LabelSmoothingCrossEntropy
from dl_models.loso_eval import evaluate_fold

OUT_ROOT = PROJECT_ROOT / "results_dl"
MODEL_DIRS = {"cnn1d": OUT_ROOT / "cnn1d", "eegnet": OUT_ROOT / "eegnet"}
MARKER_DIR = OUT_ROOT / "markers"
ERROR_DIR = OUT_ROOT / "errors"

for d in [OUT_ROOT, MARKER_DIR, ERROR_DIR, *MODEL_DIRS.values()]:
    d.mkdir(parents=True, exist_ok=True)

STEPS = [5, 10, 20, 30, 50, 75, 100, 150, 200, "full"]

# (dataset, task) pairs -- derived from CONDITION_SPEC keys, same 22
# paradigms as classical ML, in the same "DATASET_task" -> (dataset, task)
# form load_explicit() expects.
def get_paradigms():
    pairs = []
    for key in CONDITION_SPEC:
        dataset, task = key.split("_", 1)
        pairs.append((dataset, task, key))
    return sorted(pairs, key=lambda x: x[2])


def marker_path(model_name, paradigm_key, step):
    return MARKER_DIR / f"{model_name}__{paradigm_key}__step{step}.done"


def build_model(model_name, in_channels, num_classes):
    if model_name == "cnn1d":
        return ERPConv1DNet(in_channels=in_channels, num_classes=num_classes)
    elif model_name == "eegnet":
        return EEGNet(in_channels=in_channels, num_classes=num_classes)
    raise ValueError(model_name)


def subsample_indices(n_total, step, rng):
    """Same subsampling convention as Phase 3.1: random, without replacement."""
    if step == "full" or step >= n_total:
        return np.arange(n_total)
    return rng.choice(n_total, size=step, replace=False)


def run_paradigm_step(model_name, dataset, task, paradigm_key, step, device):
    marker = marker_path(model_name, paradigm_key, step)
    if marker.exists():
        print(f"[EXISTS] {model_name} {paradigm_key} step={step}", flush=True)
        return

    t0 = time.time()
    try:
        X, y, subject_ids, ch_names, sfreq = load_explicit(
            PROJECT_ROOT / "results_batch", dataset, task
        )
    except Exception as exc:
        print(f"[ERROR] load_explicit failed for {paradigm_key}: {exc}", flush=True)
        (ERROR_DIR / f"{model_name}__{paradigm_key}__step{step}.txt").write_text(str(exc))
        return

    subjects = sorted(set(subject_ids))  # deterministic, matches classical ML convention
    in_channels = X.shape[1]
    num_classes = len(np.unique(y))

    rows = []
    n_degenerate = 0
    rng = np.random.default_rng(42)

    for subject in subjects:
        test_mask = subject_ids == subject
        train_mask = ~test_mask

        X_train_full, y_train_full = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        # Apply trial-count subsampling to the TRAIN set only (test set
        # always uses all of that subject's real trials, same convention
        # as classical ML's LOSO -- the "step" limits how much training
        # data the model sees, not how much it's tested on).
        idx = subsample_indices(len(X_train_full), step, rng)
        X_train, y_train = X_train_full[idx], y_train_full[idx]

        if len(np.unique(y_train)) < 2:
            n_degenerate += 1
            continue  # cannot train on a single-class fold

        # Simple internal train/val split of the training subjects'
        # trials, purely for early stopping -- NOT a held-out subject,
        # since the true held-out test is the LOSO subject itself.
        n_val = max(1, int(0.15 * len(X_train)))
        perm = rng.permutation(len(X_train))
        val_idx, tr_idx = perm[:n_val], perm[n_val:]
        if len(np.unique(y_train[tr_idx])) < 2:
            n_degenerate += 1
            continue

        train_ds = TensorDataset(torch.from_numpy(X_train[tr_idx]).float(),
                                   torch.from_numpy(y_train[tr_idx]).long(),
                                   torch.zeros(len(tr_idx)))
        val_ds = TensorDataset(torch.from_numpy(X_train[val_idx]).float(),
                                 torch.from_numpy(y_train[val_idx]).long(),
                                 torch.zeros(len(val_idx)))
        test_ds = TensorDataset(torch.from_numpy(X_test).float(),
                                  torch.from_numpy(y_test).long(),
                                  torch.zeros(len(X_test)))

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

        model = build_model(model_name, in_channels, num_classes).to(device)
        criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

        ckpt_path = MODEL_DIRS[model_name] / f"{paradigm_key}_step{step}_{subject}.pt"
        trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion,
                           scheduler=scheduler, device=device, save_path=ckpt_path,
                           patience=8, use_amp=True)
        trainer.fit(train_loader, val_loader, epochs=30)

        model.load_state_dict(torch.load(ckpt_path, weights_only=True))
        fold_metrics = evaluate_fold(model, test_loader, device, use_amp=True)
        fold_metrics.update({"subject": subject, "n_train_trials": len(tr_idx)})
        rows.append(fold_metrics)

        ckpt_path.unlink(missing_ok=True)  # don't keep 220-paradigm x 40-subject checkpoints
        del model, trainer, optimizer, scheduler
        torch.cuda.empty_cache()

    if not rows:
        print(f"[NO DATA] {model_name} {paradigm_key} step={step}", flush=True)
        marker.touch()
        return

    out_dir = MODEL_DIRS[model_name] / paradigm_key
    out_dir.mkdir(parents=True, exist_ok=True)
    import csv
    out_csv = out_dir / f"step_{step}_folds.csv"
    with out_csv.open("w", newline="") as f:
        fieldnames = list(rows[0].keys())
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    mean_bal_acc = np.nanmean([r["balanced_accuracy"] for r in rows])
    mean_acc = np.nanmean([r["accuracy"] for r in rows])
    elapsed = time.time() - t0
    print(f"[DONE] model={model_name} paradigm={paradigm_key} step={step} "
          f"accuracy={mean_acc:.4f} balanced_accuracy={mean_bal_acc:.4f} "
          f"degenerate={n_degenerate}/{len(subjects)}", flush=True)
    print(f"[TIME] {model_name} / {paradigm_key} step={step}: {elapsed:.1f}s", flush=True)

    marker.touch()


def main():
    parser = argparse.ArgumentParser(description="Run resumable CNN-1D/EEGNet LOSO training.")
    parser.add_argument("--shard-index", type=int, default=0)
    parser.add_argument("--num-shards", type=int, default=1)
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    all_paradigms = get_paradigms()
    shard_paradigms = [p for i, p in enumerate(all_paradigms) if i % args.num_shards == args.shard_index]

    print("=" * 80, flush=True)
    print("CNN-1D / EEGNet LOSO Training", flush=True)
    print(f"Device: {device}", flush=True)
    print(f"Total paradigms: {len(all_paradigms)} | Shard: {args.shard_index}/{args.num_shards} | Assigned: {len(shard_paradigms)}", flush=True)
    print("=" * 80, flush=True)

    for dataset, task, paradigm_key in shard_paradigms:
        print(f"\\n=== {paradigm_key} ===", flush=True)
        for model_name in ["cnn1d", "eegnet"]:
            for step in STEPS:
                try:
                    run_paradigm_step(model_name, dataset, task, paradigm_key, step, device)
                except Exception as exc:
                    print(f"[ERROR] {model_name} {paradigm_key} step={step}: {exc}", flush=True)
                    (ERROR_DIR / f"{model_name}__{paradigm_key}__step{step}.txt").write_text(str(exc))

    print("\\nCNN-1D / EEGNet LOSO training complete.", flush=True)


if __name__ == "__main__":
    main()
'''

script_path = project_root / "dl_models" / "run_dl_loso.py"
script_path.write_text(script_code)
print(f"[Created] {script_path}")

[Created] /lp-dev/23BRS1328/erp_trial_min/dl_models/run_dl_loso.py


In [15]:
# ============================================================
# CELL 10: Sanity test -- one small paradigm, one step, one model,
# before launching the full 22 x 10 x 2 run. Catch bugs cheaply.
# ============================================================
import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Pick a small, fast paradigm to test with -- gonogo_1 has few subjects
# and small trial counts, so this should run in well under a minute.
dl_loso.run_paradigm_step(
    model_name="cnn1d",
    dataset="DS002680",
    task="gonogo_1",
    paradigm_key="DS002680_gonogo_1",
    step=20,
    device=device,
)

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.6827, acc: 0.588 | Val loss: 0.6946, acc: 0.333
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_step20_sub-002.pt
Epoch 002 | Train loss: 0.6661, acc: 0.588 | Val loss: 0.6958, acc: 0.333
Epoch 003 | Train loss: 0.6558, acc: 0.588 | Val loss: 0.6969, acc: 0.333
Epoch 004 | Train loss: 0.6338, acc: 0.647 | Val loss: 0.6979, acc: 0.333
Epoch 005 | Train loss: 0.6317, acc: 0.647 | Val loss: 0.6989, acc: 0.333
Epoch 006 | Train loss: 0.6135, acc: 0.765 | Val loss: 0.7002, acc: 0.333
Epoch 007 | Train loss: 0.6152, acc: 0.706 | Val loss: 0.7009, acc: 0.333
Epoch 008 | Train loss: 0.5965, acc: 0.765 | Val loss: 0.7017, acc: 0.333
Epoch 009 | Train loss: 0.6092, acc: 0.765 | Val loss: 0.7022, acc: 0.333
  [Early stopping] No improvement for 8 epochs.


TypeError: Got unsupported ScalarType BFloat16

In [16]:
# ============================================================
# CELL 11: Fix the bfloat16 -> numpy conversion bug in loso_eval.py
# ============================================================
loso_eval_path = project_root / "dl_models" / "loso_eval.py"
text = loso_eval_path.read_text()

old_block = '''            if use_amp and device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    logits = model(x)
            else:
                logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())'''

new_block = '''            if use_amp and device.type == "cuda":
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    logits = model(x)
                logits = logits.float()  # bfloat16 has no numpy equivalent --
                                          # cast back to float32 before leaving
                                          # the autocast context, not after.
            else:
                logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())'''

assert old_block in text, "Expected block not found -- paste loso_eval.py again"
text = text.replace(old_block, new_block)
loso_eval_path.write_text(text)
print("[Patched] loso_eval.py -- bfloat16 -> float32 cast fixed")

# Reload and re-run the exact same sanity test
import importlib
import dl_models.loso_eval
importlib.reload(dl_models.loso_eval)
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

# Clean up the marker/checkpoint from the failed attempt first
marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", 20)
marker.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step=20, device=device,
)

[Patched] loso_eval.py -- bfloat16 -> float32 cast fixed
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.7122, acc: 0.471 | Val loss: 0.6891, acc: 0.667
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_step20_sub-002.pt
Epoch 002 | Train loss: 0.6808, acc: 0.706 | Val loss: 0.6903, acc: 0.667
Epoch 003 | Train loss: 0.6592, acc: 0.706 | Val loss: 0.6910, acc: 0.667
Epoch 004 | Train loss: 0.6500, acc: 0.647 | Val loss: 0.6912, acc: 0.667
Epoch 005 | Train loss: 0.6288, acc: 0.647 | Val loss: 0.6912, acc: 0.667
Epoch 006 | Train loss: 0.6182, acc: 0.647 | Val loss: 0.6913, acc: 0.667
Epoch 007 | Train loss: 0.5895, acc: 0.765 | Val loss: 0.6915, acc: 0.667
Epoch 008 | Train loss: 0.6008, acc: 0.765 | Val loss: 0.6918, acc: 0.667
Epoch 009 | Train loss: 0.5728, acc: 0.765 | Val loss: 0.6922, acc: 0.667
  [Early stopping] No improvement for 

In [17]:
# ============================================================
# CELL 12: Inspect the actual per-fold CSV output
# ============================================================
import pandas as pd

csv_path = project_root / "results_dl" / "cnn1d" / "DS002680_gonogo_1" / "step_20_folds.csv"
df = pd.read_csv(csv_path)
print(df.to_string())
print(f"\nUnique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

   accuracy  balanced_accuracy        f1  auc  n_test_trials  degenerate_fold  subject  n_train_trials
0  0.270833                0.5  0.426230  0.5             48            False  sub-002              17
1  0.875000                0.5  0.933333  0.5             24            False  sub-004              17
2  0.466667                0.5  0.636364  0.5             75            False  sub-005              17
3  0.516129                0.5  0.680851  0.5             93            False  sub-006              17
4  0.489362                0.5  0.000000  0.5             94            False  sub-008              17
5  0.540230                0.5  0.000000  0.5             87            False  sub-009              17
6  0.515464                0.5  0.000000  0.5             97            False  sub-010              17
7  0.500000                0.5  0.000000  0.5             12            False  sub-012              17
8  0.510870                0.5  0.676259  0.5             92            F

In [18]:
# ============================================================
# CELL 13: Same paradigm, "full" step -- confirms whether collapse
# is data-scarcity (expected) or a real bug (still broken at full data)
# ============================================================
marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", "full")
marker.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step="full", device=device,
)

import pandas as pd
csv_path = project_root / "results_dl" / "cnn1d" / "DS002680_gonogo_1" / "step_full_folds.csv"
df = pd.read_csv(csv_path)
print(df[["subject", "accuracy", "balanced_accuracy", "auc", "n_train_trials"]].to_string())
print(f"\nUnique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.6895, acc: 0.538 | Val loss: 0.6941, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 002 | Train loss: 0.6842, acc: 0.566 | Val loss: 0.6966, acc: 0.511
Epoch 003 | Train loss: 0.6726, acc: 0.595 | Val loss: 0.7038, acc: 0.511
Epoch 004 | Train loss: 0.6655, acc: 0.616 | Val loss: 0.7127, acc: 0.511
Epoch 005 | Train loss: 0.6596, acc: 0.654 | Val loss: 2.2009, acc: 0.489
Epoch 006 | Train loss: 0.6649, acc: 0.605 | Val loss: 3.6364, acc: 0.489
Epoch 007 | Train loss: 0.6735, acc: 0.616 | Val loss: 35.1094, acc: 0.489
Epoch 008 | Train loss: 0.6712, acc: 0.609 | Val loss: 60.9431, acc: 0.489
Epoch 009 | Train loss: 0.6585, acc: 0.646 | Val loss: 6.3249, acc: 0.489
  [Early stopping] No improvement for 8 epochs.
Epoch 001 | Train loss: 0.6897, acc: 0.539 

In [19]:
# ============================================================
# CELL 14: Test whether disabling AMP fixes the BatchNorm instability
# ============================================================
script_path = project_root / "dl_models" / "run_dl_loso.py"
text = script_path.read_text()

# Disable AMP in both training and evaluation calls
text = text.replace(
    'trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion,\n                           scheduler=scheduler, device=device, save_path=ckpt_path,\n                           patience=8, use_amp=True)',
    'trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion,\n                           scheduler=scheduler, device=device, save_path=ckpt_path,\n                           patience=8, use_amp=False)'
)
text = text.replace(
    'fold_metrics = evaluate_fold(model, test_loader, device, use_amp=True)',
    'fold_metrics = evaluate_fold(model, test_loader, device, use_amp=False)'
)
script_path.write_text(text)
print("[Patched] AMP disabled for both training and evaluation")

# Re-run the exact same failing test
import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", "full")
marker.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step="full", device=device,
)

import pandas as pd
csv_path = project_root / "results_dl" / "cnn1d" / "DS002680_gonogo_1" / "step_full_folds.csv"
df = pd.read_csv(csv_path)
print(df[["subject", "accuracy", "balanced_accuracy", "auc"]].to_string())
print(f"\nUnique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

[Patched] AMP disabled for both training and evaluation
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.6911, acc: 0.587 | Val loss: 0.6938, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 002 | Train loss: 0.6569, acc: 0.654 | Val loss: 0.7265, acc: 0.511
Epoch 003 | Train loss: 0.6386, acc: 0.658 | Val loss: 0.6929, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 004 | Train loss: 0.6311, acc: 0.683 | Val loss: 0.7062, acc: 0.489
Epoch 005 | Train loss: 0.6129, acc: 0.712 | Val loss: 2.8796, acc: 0.489
Epoch 006 | Train loss: 0.5974, acc: 0.730 | Val loss: 12.5928, acc: 0.511
Epoch 007 | Train loss: 0.5915, acc: 0.740 | Val loss: 59.2080, acc: 0.489
Epoch 008 | Train loss: 0.5740, acc: 0.765 | Val loss: 96.502

In [20]:
# ============================================================
# CELL 15: Replace BatchNorm with GroupNorm in both architectures
# -- fixes the LOSO/BatchNorm cross-subject statistics mismatch
# ============================================================
cnn1d_path = project_root / "dl_models" / "cnn1d.py"
text = cnn1d_path.read_text()
backup = project_root / "dl_models" / "cnn1d_BACKUP_before_groupnorm.py"
backup.write_text(text)

text = text.replace("nn.BatchNorm1d(base_channels),", "nn.GroupNorm(8, base_channels),")
text = text.replace("nn.BatchNorm1d(base_channels * 2),", "nn.GroupNorm(8, base_channels * 2),")
text = text.replace("nn.BatchNorm1d(base_channels * 4),", "nn.GroupNorm(8, base_channels * 4),")
cnn1d_path.write_text(text)
print("[Patched] cnn1d.py -- BatchNorm1d -> GroupNorm")

eegnet_path = project_root / "dl_models" / "eegnet.py"
text = eegnet_path.read_text()
backup2 = project_root / "dl_models" / "eegnet_BACKUP_before_groupnorm.py"
backup2.write_text(text)

text = text.replace("nn.BatchNorm2d(F1),", "nn.GroupNorm(4, F1),")
text = text.replace("nn.BatchNorm2d(F1 * D),", "nn.GroupNorm(4, F1 * D),")
text = text.replace("nn.BatchNorm2d(F2),", "nn.GroupNorm(4, F2),")
eegnet_path.write_text(text)
print("[Patched] eegnet.py -- BatchNorm2d -> GroupNorm")

# Also reduce learning rate slightly as a secondary precaution --
# the exploding val loss suggests the LR is too high for this
# combination of data scale and architecture depth.
script_path = project_root / "dl_models" / "run_dl_loso.py"
text = script_path.read_text()
text = text.replace('torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)',
                     'torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)')
script_path.write_text(text)
print("[Patched] run_dl_loso.py -- lr reduced 1e-3 -> 3e-4")

# Re-run the exact same test
import importlib
import dl_models.cnn1d, dl_models.eegnet
importlib.reload(dl_models.cnn1d)
importlib.reload(dl_models.eegnet)
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", "full")
marker.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step="full", device=device,
)

import pandas as pd
csv_path = project_root / "results_dl" / "cnn1d" / "DS002680_gonogo_1" / "step_full_folds.csv"
df = pd.read_csv(csv_path)
print(df[["subject", "accuracy", "balanced_accuracy", "auc"]].to_string())
print(f"\nUnique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

[Patched] cnn1d.py -- BatchNorm1d -> GroupNorm
[Patched] eegnet.py -- BatchNorm2d -> GroupNorm
[Patched] run_dl_loso.py -- lr reduced 1e-3 -> 3e-4
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.7055, acc: 0.466 | Val loss: 0.6937, acc: 0.489
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 002 | Train loss: 0.6964, acc: 0.495 | Val loss: 0.6931, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 003 | Train loss: 0.6957, acc: 0.503 | Val loss: 0.6930, acc: 0.511
Epoch 004 | Train loss: 0.6987, acc: 0.487 | Val loss: 0.6954, acc: 0.489
Epoch 005 | Train loss: 0.6957, acc: 0.476 | Val loss: 0.6930, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.p

In [21]:
# ============================================================
# CELL 16: Isolate the variable -- keep GroupNorm, revert LR only
# ============================================================
script_path = project_root / "dl_models" / "run_dl_loso.py"
text = script_path.read_text()
text = text.replace('torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)',
                     'torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)')
script_path.write_text(text)
print("[Patched] lr reverted 3e-4 -> 1e-3 (GroupNorm kept)")

import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", "full")
marker.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step="full", device=device,
)

import pandas as pd
csv_path = project_root / "results_dl" / "cnn1d" / "DS002680_gonogo_1" / "step_full_folds.csv"
df = pd.read_csv(csv_path)
print(df[["subject", "accuracy", "balanced_accuracy", "auc"]].to_string())
print(f"\nUnique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

[Patched] lr reverted 3e-4 -> 1e-3 (GroupNorm kept)
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.7086, acc: 0.521 | Val loss: 0.6939, acc: 0.489
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 002 | Train loss: 0.6970, acc: 0.521 | Val loss: 0.6946, acc: 0.511
Epoch 003 | Train loss: 0.7007, acc: 0.509 | Val loss: 0.6965, acc: 0.511
Epoch 004 | Train loss: 0.6999, acc: 0.495 | Val loss: 0.6930, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 005 | Train loss: 0.6978, acc: 0.507 | Val loss: 0.6933, acc: 0.489
Epoch 006 | Train loss: 0.6961, acc: 0.472 | Val loss: 0.6938, acc: 0.489
Epoch 007 | Train loss: 0.6952, acc: 0.456 | Val loss: 0.6936, acc: 0.489
Epoch 008 | Train loss: 0.6938, acc: 0.489 | Val loss: 0.6933, acc:

In [22]:
# ============================================================
# CELL 17: Direct diagnosis -- inspect actual predictions on one
# real test fold, rather than guessing at more hyperparameters.
# ============================================================
import torch
import numpy as np
from dl_prep.explicit_epoch_loader import load_explicit
from dl_models.cnn1d import ERPConv1DNet
from dl_models.trainer import Trainer, LabelSmoothingCrossEntropy
from torch.utils.data import DataLoader, TensorDataset
import importlib
import dl_models.cnn1d
importlib.reload(dl_models.cnn1d)
from dl_models.cnn1d import ERPConv1DNet

X, y, subject_ids, ch_names, sfreq = load_explicit(project_root / "results_batch", "DS002680", "gonogo_1")
subjects = sorted(set(subject_ids))
test_subject = subjects[0]

test_mask = subject_ids == test_subject
train_mask = ~test_mask
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

print(f"Test subject: {test_subject}")
print(f"Train class balance: {np.bincount(y_train)}")
print(f"Test class balance: {np.bincount(y_test)}")

# Quick train (short, just to inspect behavior, not a real result)
rng = np.random.default_rng(42)
n_val = max(1, int(0.15 * len(X_train)))
perm = rng.permutation(len(X_train))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

train_ds = TensorDataset(torch.from_numpy(X_train[tr_idx]).float(), torch.from_numpy(y_train[tr_idx]).long(), torch.zeros(len(tr_idx)))
val_ds = TensorDataset(torch.from_numpy(X_train[val_idx]).float(), torch.from_numpy(y_train[val_idx]).long(), torch.zeros(len(val_idx)))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

model = ERPConv1DNet(in_channels=X.shape[1], num_classes=2).to(device)
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
ckpt_path = project_root / "results_dl" / "diag_test.pt"
trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                   device=device, save_path=ckpt_path, patience=8, use_amp=False)
trainer.fit(train_loader, val_loader, epochs=30)

model.load_state_dict(torch.load(ckpt_path, weights_only=True))
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).float().to(device))
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

print(f"\n--- Raw prediction inspection ---")
print(f"Predicted class distribution: {np.bincount(preds, minlength=2)}")
print(f"True class distribution: {np.bincount(y_test, minlength=2)}")
print(f"Sample predicted probabilities (first 10): \n{probs[:10]}")
print(f"Probability std across all test samples: class0={probs[:,0].std():.4f}, class1={probs[:,1].std():.4f}")

[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Test subject: sub-002
Train class balance: [298 303]
Test class balance: [35 13]
Epoch 001 | Train loss: 0.7173, acc: 0.495 | Val loss: 0.7067, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test.pt
Epoch 002 | Train loss: 0.6975, acc: 0.509 | Val loss: 0.6930, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test.pt
Epoch 003 | Train loss: 0.6999, acc: 0.462 | Val loss: 0.6934, acc: 0.511
Epoch 004 | Train loss: 0.6976, acc: 0.489 | Val loss: 0.6948, acc: 0.489
Epoch 005 | Train loss: 0.6966, acc: 0.477 | Val loss: 0.6930, acc: 0.511
Epoch 006 | Train loss: 0.6959, acc: 0.489 | Val loss: 0.6930, acc: 0.511
Epoch 007 | Train loss: 0.6938, acc: 0.489 | Val loss: 0.6937, acc: 0.489
Epoch 008 | Train loss: 0.6945, acc: 0.491 | Val loss: 0.6930, acc: 0.511
Epoch 009 | Train loss: 0.

In [23]:
# ============================================================
# CELL 18: Real architecture fix -- preserve temporal resolution
# instead of collapsing to a single value before classification
# ============================================================
cnn1d_path = project_root / "dl_models" / "cnn1d.py"
text = cnn1d_path.read_text()

old = '''        # Global pooling over time dimension (T) -> fixed length feature vector
        self.global_pool = nn.AdaptiveAvgPool1d(1)

        # Fully connected classification head
        self.fc1 = nn.Linear(base_channels * 4, base_channels * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(base_channels * 2, num_classes)

    def forward(self, x):
        """
        x: (N, C, T)
        """
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.global_pool(x)  # (N, channels, 1)
        x = x.squeeze(-1)        # (N, channels)

        x = self.fc1(x)'''

new = '''        # Preserve temporal resolution (pool to 8 positions, not 1) --
        # collapsing the whole trial to one value discards the timing
        # information that actually defines an ERP waveform, causing the
        # network to learn a constant, input-independent output. Verified
        # via direct prediction inspection (probability std = 0.0000
        # across all test samples with AdaptiveAvgPool1d(1)).
        POOL_POSITIONS = 8
        self.global_pool = nn.AdaptiveAvgPool1d(POOL_POSITIONS)

        # Fully connected classification head
        self.fc1 = nn.Linear(base_channels * 4 * POOL_POSITIONS, base_channels * 2)
        self.fc_dropout = nn.Dropout(dropout)
        self.fc_out = nn.Linear(base_channels * 2, num_classes)

    def forward(self, x):
        """
        x: (N, C, T)
        """
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)

        x = self.global_pool(x)  # (N, channels, POOL_POSITIONS)
        x = x.flatten(1)         # (N, channels * POOL_POSITIONS)

        x = self.fc1(x)'''

assert old in text, "Expected block not found"
text = text.replace(old, new)
cnn1d_path.write_text(text)
print("[Patched] cnn1d.py -- preserving 8 temporal positions instead of collapsing to 1")

# Same fix applied to EEGNet's global pool
eegnet_path = project_root / "dl_models" / "eegnet.py"
text = eegnet_path.read_text()
old_e = '''        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(F2, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return self.classifier(x)'''
new_e = '''        POOL_POSITIONS = 8
        self.global_pool = nn.AdaptiveAvgPool2d((1, POOL_POSITIONS))
        self.classifier = nn.Linear(F2 * POOL_POSITIONS, num_classes)

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.depthwise_conv(x)
        x = self.separable_conv(x)
        x = self.global_pool(x)
        x = x.flatten(1)
        return self.classifier(x)'''
assert old_e in text, "Expected EEGNet block not found"
text = text.replace(old_e, new_e)
eegnet_path.write_text(text)
print("[Patched] eegnet.py -- same fix applied")

# Re-run the exact diagnostic test
import importlib
import dl_models.cnn1d
importlib.reload(dl_models.cnn1d)
from dl_models.cnn1d import ERPConv1DNet

model = ERPConv1DNet(in_channels=X.shape[1], num_classes=2).to(device)
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
ckpt_path2 = project_root / "results_dl" / "diag_test2.pt"
trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                   device=device, save_path=ckpt_path2, patience=8, use_amp=False)
trainer.fit(train_loader, val_loader, epochs=30)

model.load_state_dict(torch.load(ckpt_path2, weights_only=True))
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test).float().to(device))
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

print(f"\n--- Post-fix prediction inspection ---")
print(f"Predicted class distribution: {np.bincount(preds, minlength=2)}")
print(f"True class distribution: {np.bincount(y_test, minlength=2)}")
print(f"Sample predicted probabilities (first 10):\n{probs[:10]}")
print(f"Probability std: class0={probs[:,0].std():.4f}, class1={probs[:,1].std():.4f}")

[Patched] cnn1d.py -- preserving 8 temporal positions instead of collapsing to 1
[Patched] eegnet.py -- same fix applied
Epoch 001 | Train loss: 0.7302, acc: 0.448 | Val loss: 0.6964, acc: 0.489
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test2.pt
Epoch 002 | Train loss: 0.7137, acc: 0.507 | Val loss: 0.6940, acc: 0.489
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test2.pt
Epoch 003 | Train loss: 0.7040, acc: 0.470 | Val loss: 0.6944, acc: 0.489
Epoch 004 | Train loss: 0.6950, acc: 0.513 | Val loss: 0.6931, acc: 0.511
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test2.pt
Epoch 005 | Train loss: 0.6948, acc: 0.477 | Val loss: 0.6931, acc: 0.511
Epoch 006 | Train loss: 0.6936, acc: 0.536 | Val loss: 0.6931, acc: 0.511
Epoch 007 | Train loss: 0.6958, acc: 0.460 | Val loss: 0.6935, acc: 0.489
Epoch 008 | Train loss: 0.6954, acc: 0.491 | Val loss: 0.6936, acc: 0.489
Epoch 009 |

In [24]:
# ============================================================
# CELL 19: Check the actual scale of the raw input data
# ============================================================
print(f"X dtype: {X.dtype}")
print(f"X min: {X.min():.10f}")
print(f"X max: {X.max():.10f}")
print(f"X mean: {X.mean():.10f}")
print(f"X std: {X.std():.10f}")
print(f"X abs mean: {np.abs(X).mean():.10f}")

X dtype: float32
X min: -0.0000804571
X max: 0.0000841989
X mean: -0.0000000000
X std: 0.0000081182
X abs mean: 0.0000059205


In [25]:
# ============================================================
# CELL 20: Add per-channel z-score normalization to the training
# script -- the actual root cause of tonight's collapse. Fit on
# TRAIN data only, applied to train/val/test consistently, to
# avoid any test-set leakage.
# ============================================================
script_path = project_root / "dl_models" / "run_dl_loso.py"
text = script_path.read_text()

old_block = '''        X_train_full, y_train_full = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        # Apply trial-count subsampling to the TRAIN set only (test set
        # always uses all of that subject's real trials, same convention
        # as classical ML's LOSO -- the "step" limits how much training
        # data the model sees, not how much it's tested on).
        idx = subsample_indices(len(X_train_full), step, rng)
        X_train, y_train = X_train_full[idx], y_train_full[idx]'''

new_block = '''        X_train_full, y_train_full = X[train_mask], y[train_mask]
        X_test, y_test = X[test_mask], y[test_mask]

        # Apply trial-count subsampling to the TRAIN set only (test set
        # always uses all of that subject's real trials, same convention
        # as classical ML's LOSO -- the "step" limits how much training
        # data the model sees, not how much it's tested on).
        idx = subsample_indices(len(X_train_full), step, rng)
        X_train, y_train = X_train_full[idx], y_train_full[idx]

        # Per-channel z-score normalization -- CRITICAL. Raw EEG is
        # ~1e-5 scale (real microvolt amplitudes), which is too small
        # for stable gradient-based learning with standard weight
        # initialization. Fit mean/std on the TRAIN subsample only,
        # apply identically to test -- no leakage from the held-out
        # subject into the normalization statistics.
        ch_mean = X_train.mean(axis=(0, 2), keepdims=True)
        ch_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
        X_train = (X_train - ch_mean) / ch_std
        X_test = (X_test - ch_mean) / ch_std'''

assert old_block in text, "Expected block not found"
text = text.replace(old_block, new_block)
script_path.write_text(text)
print("[Patched] run_dl_loso.py -- per-channel z-score normalization added")

# Re-run the exact same diagnostic
import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

# Manually replicate the normalization for this diagnostic cell too
ch_mean = X_train.mean(axis=(0, 2), keepdims=True)
ch_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-8
X_train_norm = (X_train - ch_mean) / ch_std
X_test_norm = (X_test - ch_mean) / ch_std

print(f"\nNormalized X_train std: {X_train_norm.std():.4f} (expect ~1.0)")

train_ds2 = TensorDataset(torch.from_numpy(X_train_norm[tr_idx]).float(), torch.from_numpy(y_train[tr_idx]).long(), torch.zeros(len(tr_idx)))
val_ds2 = TensorDataset(torch.from_numpy(X_train_norm[val_idx]).float(), torch.from_numpy(y_train[val_idx]).long(), torch.zeros(len(val_idx)))
train_loader2 = DataLoader(train_ds2, batch_size=32, shuffle=True)
val_loader2 = DataLoader(val_ds2, batch_size=64, shuffle=False)

import dl_models.cnn1d
importlib.reload(dl_models.cnn1d)
from dl_models.cnn1d import ERPConv1DNet

model = ERPConv1DNet(in_channels=X.shape[1], num_classes=2).to(device)
criterion = LabelSmoothingCrossEntropy(smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)
ckpt_path3 = project_root / "results_dl" / "diag_test3.pt"
trainer = Trainer(model=model, optimizer=optimizer, criterion=criterion, scheduler=scheduler,
                   device=device, save_path=ckpt_path3, patience=8, use_amp=False)
trainer.fit(train_loader2, val_loader2, epochs=30)

model.load_state_dict(torch.load(ckpt_path3, weights_only=True))
model.eval()
with torch.no_grad():
    logits = model(torch.from_numpy(X_test_norm).float().to(device))
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    preds = np.argmax(probs, axis=-1)

print(f"\n--- Post-normalization prediction inspection ---")
print(f"Predicted class distribution: {np.bincount(preds, minlength=2)}")
print(f"True class distribution: {np.bincount(y_test, minlength=2)}")
print(f"Sample predicted probabilities (first 10):\n{probs[:10]}")
print(f"Probability std: class0={probs[:,0].std():.4f}, class1={probs[:,1].std():.4f}")

[Patched] run_dl_loso.py -- per-channel z-score normalization added

Normalized X_train std: 0.9986 (expect ~1.0)
Epoch 001 | Train loss: 0.7191, acc: 0.564 | Val loss: 0.7403, acc: 0.522
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test3.pt
Epoch 002 | Train loss: 0.6811, acc: 0.611 | Val loss: 0.6262, acc: 0.622
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test3.pt
Epoch 003 | Train loss: 0.6049, acc: 0.748 | Val loss: 0.6160, acc: 0.667
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test3.pt
Epoch 004 | Train loss: 0.5778, acc: 0.783 | Val loss: 0.5825, acc: 0.711
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test3.pt
Epoch 005 | Train loss: 0.5492, acc: 0.793 | Val loss: 0.5735, acc: 0.744
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/diag_test3.pt
Epoch 006 | Train loss: 0.5248, acc: 0.832 | Val loss: 0

In [26]:
# ============================================================
# CELL 21: Clean up diagnostic checkpoints, run final confirmation
# through the ACTUAL orchestration script (not a manual copy)
# ============================================================
for f in (project_root / "results_dl").glob("diag_test*.pt"):
    f.unlink()
print("Diagnostic checkpoints cleaned up.")

import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

marker = dl_loso.marker_path("cnn1d", "DS002680_gonogo_1", "full")
marker.unlink(missing_ok=True)
out_csv = dl_loso.MODEL_DIRS["cnn1d"] / "DS002680_gonogo_1" / "step_full_folds.csv"
out_csv.unlink(missing_ok=True)

dl_loso.run_paradigm_step(
    model_name="cnn1d", dataset="DS002680", task="gonogo_1",
    paradigm_key="DS002680_gonogo_1", step="full", device=device,
)

import pandas as pd
df = pd.read_csv(out_csv)
print(df[["subject", "accuracy", "balanced_accuracy", "auc"]].to_string())
print(f"\nMean balanced accuracy: {df['balanced_accuracy'].mean():.4f}")
print(f"Unique balanced_accuracy values: {df['balanced_accuracy'].nunique()} out of {len(df)}")

Diagnostic checkpoints cleaned up.
[Explicit Loader] DS002680_gonogo_1: X=(649, 31, 1001), y=(649,), classes=[0 1], subjects=10, class_balance=[333, 316]
Epoch 001 | Train loss: 0.7440, acc: 0.571 | Val loss: 0.6837, acc: 0.578
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 002 | Train loss: 0.6634, acc: 0.640 | Val loss: 0.6054, acc: 0.689
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 003 | Train loss: 0.6013, acc: 0.736 | Val loss: 0.6016, acc: 0.689
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull_sub-002.pt
Epoch 004 | Train loss: 0.5776, acc: 0.746 | Val loss: 0.6119, acc: 0.711
Epoch 005 | Train loss: 0.5676, acc: 0.763 | Val loss: 0.5791, acc: 0.744
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS002680_gonogo_1_stepfull

In [27]:
# ============================================================
# CELL 22: Launch the full 22-paradigm x 10-step x 2-architecture
# background training run
# ============================================================
import subprocess

log_path = project_root / "logs_dl_loso.log"
cmd = ["python3", "dl_models/run_dl_loso.py", "--shard-index", "0", "--num-shards", "1"]

with open(log_path, "w") as logf:
    proc = subprocess.Popen(cmd, cwd=str(project_root), stdout=logf, stderr=subprocess.STDOUT)

print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=2853484
Log: /lp-dev/23BRS1328/erp_trial_min/logs_dl_loso.log


In [28]:
import time
time.sleep(45)
print((project_root / "logs_dl_loso.log").read_text()[-3000:])

ckpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS002680_gonogo_1_step150_sub-014.pt
Epoch 014 | Train loss: 0.6523, acc: 0.656 | Val loss: 0.6143, acc: 0.727
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS002680_gonogo_1_step150_sub-014.pt
Epoch 015 | Train loss: 0.6522, acc: 0.680 | Val loss: 0.6103, acc: 0.773
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS002680_gonogo_1_step150_sub-014.pt
Epoch 016 | Train loss: 0.6297, acc: 0.695 | Val loss: 0.6081, acc: 0.773
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS002680_gonogo_1_step150_sub-014.pt
Epoch 017 | Train loss: 0.6356, acc: 0.664 | Val loss: 0.6071, acc: 0.727
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS002680_gonogo_1_step150_sub-014.pt
Epoch 018 | Train loss: 0.6325, acc: 0.664 | Val loss: 0.6074, acc: 0.727
Epoch 019 | Train loss: 0.

In [29]:
import time
time.sleep(120)
log_text = (project_root / "logs_dl_loso.log").read_text()

# Just show the [DONE] lines -- summary view, not full epoch-by-epoch noise
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

=== DS002680_gonogo_1 ===
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=5 accuracy=0.3945 balanced_accuracy=0.4794 degenerate=2/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=10 accuracy=0.5219 balanced_accuracy=0.5041 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=30 accuracy=0.5008 balanced_accuracy=0.5196 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=50 accuracy=0.5869 balanced_accuracy=0.5859 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=75 accuracy=0.5017 balanced_accuracy=0.5245 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=100 accuracy=0.5782 balanced_accuracy=0.6106 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=150 accuracy=0.6083 balanced_accuracy=0.5847 degenerate=0/10
[DONE] model=cnn1d paradigm=DS002680_gonogo_1 step=200 accuracy=0.5807 balanced_accuracy=0.5614 degenerate=0/10
[DONE] model=eegnet paradigm=DS002680_gonogo_1 step=5 accuracy=0.5438 balanced_accur

In [30]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

Progress: 49 / 440


In [31]:
log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=5 accuracy=0.5798 balanced_accuracy=0.5161 degenerate=1/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=10 accuracy=0.4557 balanced_accuracy=0.4900 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=20 accuracy=0.5302 balanced_accuracy=0.5363 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=30 accuracy=0.5507 balanced_accuracy=0.5230 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=50 accuracy=0.6464 balanced_accuracy=0.6285 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=75 accuracy=0.6282 balanced_accuracy=0.6104 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=100 accuracy=0.6563 balanced_accuracy=0.6497 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=150 accuracy=0.6836 balanced_accuracy=0.6645 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_10 step=200 accuracy=0.6537 balanced_accuracy=0.6692 degene

In [32]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

Progress: 67 / 440


In [33]:
log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nProgress: {len(markers)} / 440")

[DONE] model=cnn1d paradigm=DS002680_gonogo_8 step=150 accuracy=0.9400 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_8 step=200 accuracy=0.9557 balanced_accuracy=0.9560 degenerate=0/12
[DONE] model=cnn1d paradigm=DS002680_gonogo_8 step=full accuracy=0.9539 balanced_accuracy=0.9547 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=5 accuracy=0.7408 balanced_accuracy=0.7381 degenerate=2/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenera

In [34]:
log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nProgress: {len(markers)} / 440")

[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=100 accuracy=0.8611 balanced_accuracy=0.8641 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=150 accuracy=0.9008 balanced_accuracy=0.9023 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=200 accuracy=0.9411 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=full accuracy=0.9718 balanced_accuracy=0.9729 deg

In [36]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

Progress: 269 / 440
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=100 accuracy=0.8611 balanced_accuracy=0.8641 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=150 accuracy=0.9008 balanced_accuracy=0.9023 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=200 accuracy=0.9411 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=full accuracy=0.9718 balanced

In [37]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

Progress: 269 / 440
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=100 accuracy=0.8611 balanced_accuracy=0.8641 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=150 accuracy=0.9008 balanced_accuracy=0.9023 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=200 accuracy=0.9411 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=full accuracy=0.9718 balanced

In [38]:
import time
print(f"Log file last modified: {time.ctime((project_root / 'logs_dl_loso.log').stat().st_mtime)}")
print(f"Current time: {time.ctime()}")

Log file last modified: Sun Aug 30 16:01:50 2026
Current time: Sun Aug 30 16:02:26 2026


In [39]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

Progress: 269 / 440
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=100 accuracy=0.8611 balanced_accuracy=0.8641 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=150 accuracy=0.9008 balanced_accuracy=0.9023 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=200 accuracy=0.9411 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=full accuracy=0.9718 balanced

In [40]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-40:]))

Progress: 269 / 440
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=10 accuracy=0.5857 balanced_accuracy=0.5961 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=20 accuracy=0.8224 balanced_accuracy=0.8230 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=30 accuracy=0.7697 balanced_accuracy=0.7714 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=50 accuracy=0.8189 balanced_accuracy=0.8209 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=75 accuracy=0.8477 balanced_accuracy=0.8512 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=100 accuracy=0.8611 balanced_accuracy=0.8641 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=150 accuracy=0.9008 balanced_accuracy=0.9023 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=200 accuracy=0.9411 balanced_accuracy=0.9427 degenerate=0/12
[DONE] model=eegnet paradigm=DS002680_gonogo_8 step=full accuracy=0.9718 balanced

In [41]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

Progress: 269 / 440


In [42]:
import time
print(f"Current time: {time.ctime()}")
print(f"Markers dir last modified: {time.ctime((project_root / 'results_dl' / 'markers').stat().st_mtime)}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

# Also show the most recently created markers, sorted by time
recent = sorted(markers, key=lambda m: m.stat().st_mtime, reverse=True)[:5]
for m in recent:
    print(f"  {m.name}: {time.ctime(m.stat().st_mtime)}")

Current time: Sun Aug 30 16:13:13 2026
Markers dir last modified: Sun Aug 30 14:48:41 2026
Progress: 269 / 440
  cnn1d__DS006018_auditoryoddball__step200.done: Sun Aug 30 14:48:41 2026
  cnn1d__DS006018_auditoryoddball__step150.done: Sun Aug 30 14:47:08 2026
  cnn1d__DS006018_auditoryoddball__step100.done: Sun Aug 30 14:45:31 2026
  cnn1d__DS006018_auditoryoddball__step75.done: Sun Aug 30 14:43:45 2026
  cnn1d__DS006018_auditoryoddball__step50.done: Sun Aug 30 14:42:11 2026


In [43]:
import subprocess
result = subprocess.run(["ps", "-p", "2853484", "-o", "pid,pcpu,pmem,etime,stat,cmd"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

    PID %CPU %MEM     ELAPSED STAT CMD
2853484  109  0.5    02:24:09 Rl   python3 dl_models/run_dl_loso.py --shard-index 0 --num-shards 1




In [44]:
# Check the actual last lines of the log to see what step/epoch it's on right now
log_text = (project_root / "logs_dl_loso.log").read_text()
print(log_text[-2000:])

0
Epoch 014 | Train loss: 0.5513, acc: 0.792 | Val loss: 0.5800, acc: 0.790
Epoch 015 | Train loss: 0.5462, acc: 0.793 | Val loss: 0.5823, acc: 0.788
Epoch 016 | Train loss: 0.5398, acc: 0.795 | Val loss: 0.5920, acc: 0.774
  [Early stopping] No improvement for 8 epochs.
Epoch 001 | Train loss: 0.5857, acc: 0.791 | Val loss: 0.5812, acc: 0.788
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_auditoryoddball_stepfull_sub-096.pt
Epoch 002 | Train loss: 0.5766, acc: 0.791 | Val loss: 0.5759, acc: 0.788
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_auditoryoddball_stepfull_sub-096.pt
Epoch 003 | Train loss: 0.5741, acc: 0.791 | Val loss: 0.5749, acc: 0.788
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_auditoryoddball_stepfull_sub-096.pt
Epoch 004 | Train loss: 0.5717, acc: 0.791 | Val loss: 0.5750, acc: 0.788
Epoch 005 | Train loss: 0.5710, acc: 0.791 | 

In [45]:
log_text = (project_root / "logs_dl_loso.log").read_text()
lines = log_text.splitlines()

# Find the most recent "Saved best model" line -- tells us which subject
# is currently being processed, a better real-time progress indicator
save_lines = [l for l in lines if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

# Also show marker count for the overall picture
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_auditoryoddball_stepfull_sub-107.pt

Completed steps (markers): 269 / 440


In [47]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_auditoryoddball_stepfull_sub-117.pt

Completed steps (markers): 269 / 440


In [48]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-021.pt

Completed steps (markers): 279 / 440


In [49]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-033.pt

Completed steps (markers): 279 / 440


In [50]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-048.pt

Completed steps (markers): 279 / 440


In [51]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-061.pt

Completed steps (markers): 279 / 440


In [52]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-090.pt

Completed steps (markers): 279 / 440


In [53]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_auditoryoddball_stepfull_sub-096.pt

Completed steps (markers): 279 / 440


In [54]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/DS006018_visualoddball_stepfull_sub-024.pt

Completed steps (markers): 289 / 440


In [55]:
log_text = (project_root / "logs_dl_loso.log").read_text()
save_lines = [l for l in log_text.splitlines() if "Saved best model" in l]
if save_lines:
    print(f"Most recent checkpoint: {save_lines[-1]}")

markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nCompleted steps (markers): {len(markers)} / 440")

Most recent checkpoint:   [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/eegnet/DS006018_visualoddball_stepfull_sub-125.pt

Completed steps (markers): 300 / 440


In [56]:
import subprocess
import time

# Check if the process is still alive
result = subprocess.run(["ps", "-p", "2853484", "-o", "pid,pcpu,pmem,etime,stat,cmd"], capture_output=True, text=True)
print("=== Process check ===")
print(result.stdout)
print(result.stderr)

# Check how recently the log was actually written to
log_path = project_root / "logs_dl_loso.log"
print(f"\n=== Log freshness ===")
print(f"Current time: {time.ctime()}")
print(f"Log last modified: {time.ctime(log_path.stat().st_mtime)}")

# Real marker count, right now
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"\nProgress: {len(markers)} / 440")

=== Process check ===
    PID %CPU %MEM     ELAPSED STAT CMD
2853484 55.7  0.0    14:41:24 Z    [python3] <defunct>



=== Log freshness ===
Current time: Mon Aug 31 04:30:58 2026
Log last modified: Sun Aug 30 21:40:00 2026

Progress: 300 / 440


In [57]:
# ============================================================
# Resume the CNN-1D/EEGNet training from where it died.
# Marker-based resumability means this correctly skips the
# 300 already-completed combinations.
# ============================================================
import subprocess

# Clean up the dead zombie reference (harmless, just tidiness)
subprocess.run(["kill", "-9", "2853484"], capture_output=True)

log_path = project_root / "logs_dl_loso_resumed.log"
cmd = ["python3", "dl_models/run_dl_loso.py", "--shard-index", "0", "--num-shards", "1"]

with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        cmd, cwd=str(project_root),
        stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,  # detach fully from this shell/kernel session
    )

print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=1497527
Log: /lp-dev/23BRS1328/erp_trial_min/logs_dl_loso_resumed.log


In [58]:
import time
time.sleep(45)
print((project_root / "logs_dl_loso_resumed.log").read_text()[-2500:])

or ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400
[ERROR] load_explicit failed for ERP_CORE_N400: No subject folders found for ERP_CORE_N400

=== ERP_CORE_P3 ===
[ERROR] load_explicit failed for ERP_CORE_P3: No subject folders found for ERP_CORE_P3
[ERROR] load_explicit failed for ERP_CORE_P3: No subject folders found for ERP_CORE_P3
[ERROR] load_explicit failed for ERP_CORE_P3: No subject folders found for ERP_CORE_P3
[ERROR] load_explici

In [59]:
from pathlib import Path
results_batch = project_root / "results_batch"

print(f"results_batch exists: {results_batch.exists()}")
print(f"ERP_CORE exists: {(results_batch / 'ERP_CORE').exists()}")

erp_core_dir = results_batch / "ERP_CORE"
if erp_core_dir.exists():
    subject_dirs = sorted(erp_core_dir.glob("sub-*"))
    print(f"Subject folders found: {len(subject_dirs)}")
    if subject_dirs:
        print(f"Sample: {subject_dirs[0]}")
        n400_path = subject_dirs[0] / "N400"
        print(f"N400 subfolder for {subject_dirs[0].name}: exists={n400_path.exists()}")
else:
    print("ERP_CORE directory itself is missing!")
    print(f"\nWhat IS in results_batch/:")
    for item in results_batch.iterdir():
        print(f"  {item.name}")

results_batch exists: True
ERP_CORE exists: True
Subject folders found: 40
Sample: /lp-dev/23BRS1328/erp_trial_min/results_batch/ERP_CORE/sub-001
N400 subfolder for sub-001: exists=True


In [60]:
# Check the actual glob pattern being used in load_explicit / explicit_epoch_loader
import inspect
from dl_prep.explicit_epoch_loader import load_explicit
print(inspect.getsource(load_explicit))

def load_explicit(results_batch_dir: Path, dataset_name: str, task_name: str):
    """
    Loads exactly the trials CONDITION_SPEC says belong to this task's
    binary classification, from real subject folders, with no globbing
    of unlisted files and no keyword inference.
    """
    spec_key = f"{dataset_name}_{task_name}"
    if spec_key not in CONDITION_SPEC:
        raise ValueError(
            f"'{spec_key}' has no entry in CONDITION_SPEC. Refusing to load with a "
            f"fallback -- this is exactly the failure mode that caused undetected "
            f"issues this session. Add an explicit, verified entry before loading."
        )
    spec = CONDITION_SPEC[spec_key]

    task_dirs = sorted((results_batch_dir / dataset_name).glob(f"sub-*/{task_name}"))
    if not task_dirs:
        raise FileNotFoundError(f"No subject folders found for {spec_key}")

    X_list, y_list, sub_list = [], [], []
    ch_names, sfreq = None, None

    for tdir in task_dirs:
        sub_name 

In [61]:
# ============================================================
# Fix get_paradigms() -- match against known dataset prefixes
# explicitly, rather than splitting on the first underscore
# (which breaks for "ERP_CORE", the one dataset name that itself
# contains an underscore).
# ============================================================
script_path = project_root / "dl_models" / "run_dl_loso.py"
text = script_path.read_text()

old_func = '''def get_paradigms():
    pairs = []
    for key in CONDITION_SPEC:
        dataset, task = key.split("_", 1)
        pairs.append((dataset, task, key))
    return sorted(pairs, key=lambda x: x[2])'''

new_func = '''def get_paradigms():
    # Match against KNOWN dataset names explicitly -- splitting on the
    # first underscore breaks for "ERP_CORE", which itself contains an
    # underscore (this caused all 7 ERP_CORE paradigms to silently fail
    # with "no subject folders found" once the run reached them).
    KNOWN_DATASETS = ["ERP_CORE", "DS006018", "DS002680"]
    pairs = []
    for key in CONDITION_SPEC:
        matched = False
        for ds in KNOWN_DATASETS:
            if key.startswith(ds + "_"):
                task = key[len(ds) + 1:]
                pairs.append((ds, task, key))
                matched = True
                break
        if not matched:
            raise ValueError(f"Paradigm key '{key}' doesn't match any known dataset prefix")
    return sorted(pairs, key=lambda x: x[2])'''

assert old_func in text, "Expected function not found"
text = text.replace(old_func, new_func)
script_path.write_text(text)
print("[Patched] get_paradigms() -- explicit dataset-name matching")

# Verify the fix directly
import importlib
import dl_models.run_dl_loso as dl_loso
importlib.reload(dl_loso)

for dataset, task, key in dl_loso.get_paradigms():
    if "ERP_CORE" in key:
        print(f"dataset={dataset!r}, task={task!r}, key={key!r}")

[Patched] get_paradigms() -- explicit dataset-name matching
dataset='ERP_CORE', task='ERN', key='ERP_CORE_ERN'
dataset='ERP_CORE', task='LRP', key='ERP_CORE_LRP'
dataset='ERP_CORE', task='MMN', key='ERP_CORE_MMN'
dataset='ERP_CORE', task='N170', key='ERP_CORE_N170'
dataset='ERP_CORE', task='N2pc', key='ERP_CORE_N2pc'
dataset='ERP_CORE', task='N400', key='ERP_CORE_N400'
dataset='ERP_CORE', task='P3', key='ERP_CORE_P3'


In [62]:
# ============================================================
# Relaunch the training run with the fix applied. Marker-based
# resumability means the 300 already-completed combinations
# are correctly skipped; only the 140 ERP_CORE combinations
# (which never actually ran due to the bug) will execute.
# ============================================================
import subprocess

log_path = project_root / "logs_dl_loso_resumed2.log"
cmd = ["python3", "dl_models/run_dl_loso.py", "--shard-index", "0", "--num-shards", "1"]

with open(log_path, "w") as logf:
    proc = subprocess.Popen(
        cmd, cwd=str(project_root),
        stdout=logf, stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print(f"Started, PID={proc.pid}")
print(f"Log: {log_path}")

Started, PID=1512392
Log: /lp-dev/23BRS1328/erp_trial_min/logs_dl_loso_resumed2.log


In [63]:
import time
time.sleep(45)
print((project_root / "logs_dl_loso_resumed2.log").read_text()[-2500:])

erbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py:130: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py:130: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py:130: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py:130: RuntimeWarning: The events passed to the Epochs constructor are not chronologically ordered.
  epochs = mne.read_epochs(fpath, preload=True, verbose=False)
/lp-dev/23BRS1328/erp_trial_min/dl_prep/explicit_epoch_loader.py:130: RuntimeWarni

In [64]:
log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
lines = log_text.splitlines()

# Filter out the cosmetic MNE warning lines, keep everything else
real_lines = [l for l in lines if "RuntimeWarning" not in l and "read_epochs" not in l]
print("\n".join(real_lines[-40:]))

Epoch 006 | Train loss: 0.6694, acc: 0.624 | Val loss: 0.7065, acc: 0.467
Epoch 007 | Train loss: 0.6649, acc: 0.612 | Val loss: 0.7437, acc: 0.467
Epoch 008 | Train loss: 0.6420, acc: 0.671 | Val loss: 0.7029, acc: 0.467
Epoch 009 | Train loss: 0.6098, acc: 0.753 | Val loss: 0.7143, acc: 0.533
Epoch 010 | Train loss: 0.5733, acc: 0.824 | Val loss: 0.7661, acc: 0.467
Epoch 011 | Train loss: 0.5638, acc: 0.765 | Val loss: 0.8426, acc: 0.400
Epoch 012 | Train loss: 0.5491, acc: 0.776 | Val loss: 0.8328, acc: 0.467
  [Early stopping] No improvement for 8 epochs.
Epoch 001 | Train loss: 0.7694, acc: 0.518 | Val loss: 0.7647, acc: 0.467
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/ERP_CORE_ERN_step100_sub-038.pt
Epoch 002 | Train loss: 0.7460, acc: 0.506 | Val loss: 0.7668, acc: 0.533
Epoch 003 | Train loss: 0.6604, acc: 0.600 | Val loss: 0.7344, acc: 0.467
  [Checkpoint] Saved best model -> /lp-dev/23BRS1328/erp_trial_min/results_dl/cnn1d/ERP_CORE_ERN

In [65]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 309 / 440
=== DS002680_gonogo_2 ===
=== DS002680_gonogo_3 ===
=== DS002680_gonogo_4 ===
=== DS002680_gonogo_5 ===
=== DS002680_gonogo_6 ===
=== DS002680_gonogo_7 ===
=== DS002680_gonogo_8 ===
=== DS002680_gonogo_9 ===
=== DS006018_auditoryoddball ===
=== DS006018_visualoddball ===
=== ERP_CORE_ERN ===
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=5 accuracy=0.5031 balanced_accuracy=0.5175 degenerate=9/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=10 accuracy=0.5033 balanced_accuracy=0.4879 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=20 accuracy=0.4966 balanced_accuracy=0.5084 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=30 accuracy=0.5069 balanced_accuracy=0.5204 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=50 accuracy=0.5198 balanced_accuracy=0.5189 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=75 accuracy=0.5590 balanced_accuracy=0.5489 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=100 ac

In [66]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 319 / 440
=== ERP_CORE_ERN ===
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=5 accuracy=0.5031 balanced_accuracy=0.5175 degenerate=9/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=10 accuracy=0.5033 balanced_accuracy=0.4879 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=20 accuracy=0.4966 balanced_accuracy=0.5084 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=30 accuracy=0.5069 balanced_accuracy=0.5204 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=50 accuracy=0.5198 balanced_accuracy=0.5189 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=75 accuracy=0.5590 balanced_accuracy=0.5489 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=100 accuracy=0.5298 balanced_accuracy=0.5288 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=150 accuracy=0.5673 balanced_accuracy=0.5654 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_ERN step=200 accuracy=0.5819 balanced_accuracy=0.5845 degenerate=0/37
[DO

In [67]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 329 / 440
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=5 accuracy=0.4881 balanced_accuracy=0.5009 degenerate=9/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=10 accuracy=0.5046 balanced_accuracy=0.5059 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=20 accuracy=0.5274 balanced_accuracy=0.5105 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=30 accuracy=0.5296 balanced_accuracy=0.5309 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=50 accuracy=0.5368 balanced_accuracy=0.5267 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=75 accuracy=0.5138 balanced_accuracy=0.5150 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=100 accuracy=0.5078 balanced_accuracy=0.5121 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=150 accuracy=0.5580 balanced_accuracy=0.5567 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=200 accuracy=0.5868 balanced_accuracy=0.5792 degenerate=0/37
[DONE] model=ee

In [68]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 329 / 440
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=5 accuracy=0.4881 balanced_accuracy=0.5009 degenerate=9/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=10 accuracy=0.5046 balanced_accuracy=0.5059 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=20 accuracy=0.5274 balanced_accuracy=0.5105 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=30 accuracy=0.5296 balanced_accuracy=0.5309 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=50 accuracy=0.5368 balanced_accuracy=0.5267 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=75 accuracy=0.5138 balanced_accuracy=0.5150 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=100 accuracy=0.5078 balanced_accuracy=0.5121 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=150 accuracy=0.5580 balanced_accuracy=0.5567 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=200 accuracy=0.5868 balanced_accuracy=0.5792 degenerate=0/37
[DONE] model=ee

In [69]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 329 / 440
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=5 accuracy=0.4881 balanced_accuracy=0.5009 degenerate=9/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=10 accuracy=0.5046 balanced_accuracy=0.5059 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=20 accuracy=0.5274 balanced_accuracy=0.5105 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=30 accuracy=0.5296 balanced_accuracy=0.5309 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=50 accuracy=0.5368 balanced_accuracy=0.5267 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=75 accuracy=0.5138 balanced_accuracy=0.5150 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=100 accuracy=0.5078 balanced_accuracy=0.5121 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=150 accuracy=0.5580 balanced_accuracy=0.5567 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=200 accuracy=0.5868 balanced_accuracy=0.5792 degenerate=0/37
[DONE] model=ee

In [70]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 333 / 440
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=50 accuracy=0.5368 balanced_accuracy=0.5267 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=75 accuracy=0.5138 balanced_accuracy=0.5150 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=100 accuracy=0.5078 balanced_accuracy=0.5121 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=150 accuracy=0.5580 balanced_accuracy=0.5567 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=200 accuracy=0.5868 balanced_accuracy=0.5792 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_ERN step=full accuracy=0.7953 balanced_accuracy=0.7953 degenerate=0/37
=== ERP_CORE_LRP ===
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=5 accuracy=0.5843 balanced_accuracy=0.5774 degenerate=5/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=10 accuracy=0.6251 balanced_accuracy=0.6203 degenerate=1/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=20 accuracy=0.6836 balanced_accuracy=0.6734 degenerate=

In [71]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 339 / 440
=== ERP_CORE_LRP ===
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=5 accuracy=0.5843 balanced_accuracy=0.5774 degenerate=5/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=10 accuracy=0.6251 balanced_accuracy=0.6203 degenerate=1/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=20 accuracy=0.6836 balanced_accuracy=0.6734 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=30 accuracy=0.7386 balanced_accuracy=0.7366 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=50 accuracy=0.8412 balanced_accuracy=0.8344 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=75 accuracy=0.8373 balanced_accuracy=0.8353 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=100 accuracy=0.8721 balanced_accuracy=0.8695 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=150 accuracy=0.8810 balanced_accuracy=0.8796 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=200 accuracy=0.8917 balanced_accuracy=0.8882 degenerate=0/37
[DO

In [72]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 339 / 440
=== ERP_CORE_LRP ===
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=5 accuracy=0.5843 balanced_accuracy=0.5774 degenerate=5/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=10 accuracy=0.6251 balanced_accuracy=0.6203 degenerate=1/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=20 accuracy=0.6836 balanced_accuracy=0.6734 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=30 accuracy=0.7386 balanced_accuracy=0.7366 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=50 accuracy=0.8412 balanced_accuracy=0.8344 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=75 accuracy=0.8373 balanced_accuracy=0.8353 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=100 accuracy=0.8721 balanced_accuracy=0.8695 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=150 accuracy=0.8810 balanced_accuracy=0.8796 degenerate=0/37
[DONE] model=cnn1d paradigm=ERP_CORE_LRP step=200 accuracy=0.8917 balanced_accuracy=0.8882 degenerate=0/37
[DO

In [73]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 349 / 440
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=5 accuracy=0.5164 balanced_accuracy=0.5112 degenerate=5/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=10 accuracy=0.5886 balanced_accuracy=0.5844 degenerate=1/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=20 accuracy=0.6208 balanced_accuracy=0.6159 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=30 accuracy=0.6894 balanced_accuracy=0.6878 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=50 accuracy=0.7491 balanced_accuracy=0.7473 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=75 accuracy=0.8015 balanced_accuracy=0.7987 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=100 accuracy=0.8346 balanced_accuracy=0.8326 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=150 accuracy=0.8461 balanced_accuracy=0.8446 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=200 accuracy=0.8782 balanced_accuracy=0.8750 degenerate=0/37
[DONE] model=ee

In [74]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 349 / 440
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=5 accuracy=0.5164 balanced_accuracy=0.5112 degenerate=5/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=10 accuracy=0.5886 balanced_accuracy=0.5844 degenerate=1/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=20 accuracy=0.6208 balanced_accuracy=0.6159 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=30 accuracy=0.6894 balanced_accuracy=0.6878 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=50 accuracy=0.7491 balanced_accuracy=0.7473 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=75 accuracy=0.8015 balanced_accuracy=0.7987 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=100 accuracy=0.8346 balanced_accuracy=0.8326 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=150 accuracy=0.8461 balanced_accuracy=0.8446 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_LRP step=200 accuracy=0.8782 balanced_accuracy=0.8750 degenerate=0/37
[DONE] model=ee

In [75]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 359 / 440
=== ERP_CORE_MMN ===
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=5 accuracy=0.6651 balanced_accuracy=0.4975 degenerate=15/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=10 accuracy=0.7236 balanced_accuracy=0.4998 degenerate=2/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=20 accuracy=0.7737 balanced_accuracy=0.4973 degenerate=1/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=30 accuracy=0.7889 balanced_accuracy=0.4999 degenerate=0/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=50 accuracy=0.7867 balanced_accuracy=0.4982 degenerate=0/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=75 accuracy=0.7854 balanced_accuracy=0.5000 degenerate=0/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=100 accuracy=0.7937 balanced_accuracy=0.5012 degenerate=0/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=150 accuracy=0.7899 balanced_accuracy=0.5006 degenerate=0/40
[DONE] model=cnn1d paradigm=ERP_CORE_MMN step=200 accuracy=0.7949 balanced_accuracy=0.5007 degenerate=0/40
[D

In [79]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 390 / 440
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=10 accuracy=0.4763 balanced_accuracy=0.4938 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=20 accuracy=0.4733 balanced_accuracy=0.4974 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=30 accuracy=0.5016 balanced_accuracy=0.5061 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=50 accuracy=0.5134 balanced_accuracy=0.4947 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=75 accuracy=0.4994 balanced_accuracy=0.5130 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=100 accuracy=0.5256 balanced_accuracy=0.5159 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=150 accuracy=0.5025 balanced_accuracy=0.5142 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=200 accuracy=0.5574 balanced_accuracy=0.5570 degenerate=0/37
[DONE] model=eegnet paradigm=ERP_CORE_N170 step=full accuracy=0.6898 balanced_accuracy=0.6849 degenerate=0/37
===

In [80]:
markers = list((project_root / "results_dl" / "markers").glob("*.done"))
print(f"Progress: {len(markers)} / 440")

log_text = (project_root / "logs_dl_loso_resumed2.log").read_text()
done_lines = [l for l in log_text.splitlines() if "[DONE]" in l or "===" in l]
print("\n".join(done_lines[-20:]))

Progress: 440 / 440
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=5 accuracy=0.7022 balanced_accuracy=0.4721 degenerate=15/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=10 accuracy=0.7576 balanced_accuracy=0.5091 degenerate=4/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=20 accuracy=0.7429 balanced_accuracy=0.5047 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=30 accuracy=0.7898 balanced_accuracy=0.4990 degenerate=1/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=50 accuracy=0.7667 balanced_accuracy=0.5084 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=75 accuracy=0.7868 balanced_accuracy=0.4980 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=100 accuracy=0.7884 balanced_accuracy=0.4992 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=150 accuracy=0.7733 balanced_accuracy=0.4864 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_CORE_P3 step=200 accuracy=0.7869 balanced_accuracy=0.4972 degenerate=0/36
[DONE] model=cnn1d paradigm=ERP_

In [81]:
# ============================================================
# Full verification pass across all 22 paradigms, both models
# ============================================================
import pandas as pd
from pathlib import Path

results = []
for model in ["cnn1d", "eegnet"]:
    model_dir = project_root / "results_dl" / model
    for paradigm_dir in sorted(model_dir.glob("*")):
        if not paradigm_dir.is_dir():
            continue
        full_csv = paradigm_dir / "step_full_folds.csv"
        if full_csv.exists():
            df = pd.read_csv(full_csv)
            results.append({
                "model": model,
                "paradigm": paradigm_dir.name,
                "mean_balanced_acc": df["balanced_accuracy"].mean(),
                "n_folds": len(df),
                "unique_bal_acc": df["balanced_accuracy"].nunique(),
            })

summary_df = pd.DataFrame(results).sort_values(["paradigm", "model"])
pd.set_option("display.max_rows", 50)
print(summary_df.to_string(index=False))

print(f"\nTotal paradigm/model combos with full-step results: {len(summary_df)} (expect 44)")
print(f"Combos where balanced_accuracy is suspiciously frozen (unique==1): {(summary_df['unique_bal_acc']==1).sum()}")

 model                 paradigm  mean_balanced_acc  n_folds  unique_bal_acc
 cnn1d        DS002680_gonogo_1           0.651741       10              10
eegnet        DS002680_gonogo_1           0.581517       10              10
 cnn1d       DS002680_gonogo_10           0.701608       12              12
eegnet       DS002680_gonogo_10           0.678956       12              12
 cnn1d       DS002680_gonogo_11           0.936697       12               9
eegnet       DS002680_gonogo_11           0.927826       12              10
 cnn1d       DS002680_gonogo_12           0.963709       11               5
eegnet       DS002680_gonogo_12           0.946071       11               6
 cnn1d       DS002680_gonogo_13           0.635751       11              11
eegnet       DS002680_gonogo_13           0.687281       11              11
 cnn1d        DS002680_gonogo_2           0.963594       11               6
eegnet        DS002680_gonogo_2           0.941943       11               7
 cnn1d      

In [82]:
# ============================================================
# Classical ML's full-trial-count-ONLY average -- for a fair,
# apples-to-apples comparison against CNN-1D/EEGNet's full-step
# average (0.7392 / 0.7395)
# ============================================================
import pandas as pd
from pathlib import Path

classical_results = []
for model in ["random_forest", "xgboost", "svm"]:
    model_dir = project_root / "results_stage4_classical_ml" / model
    for paradigm_dir in sorted(model_dir.glob("*")):
        if not paradigm_dir.is_dir():
            continue
        full_csv = paradigm_dir / "step_full_folds.csv"
        if full_csv.exists():
            df = pd.read_csv(full_csv)
            classical_results.append({
                "model": model,
                "paradigm": paradigm_dir.name,
                "mean_balanced_acc": df["balanced_accuracy"].mean(),
                "n_folds": len(df),
            })

classical_df = pd.DataFrame(classical_results)
print(classical_df.to_string(index=False))

print(f"\n{'='*60}")
print("FULL-TRIAL-COUNT-ONLY AVERAGES (fair comparison basis)")
print(f"{'='*60}")
for model in ["random_forest", "xgboost", "svm"]:
    subset = classical_df[classical_df["model"] == model]
    print(f"{model}: {subset['mean_balanced_acc'].mean():.4f} (n={len(subset)} paradigms)")

overall_classical_full = classical_df["mean_balanced_acc"].mean()
print(f"\nAll classical models combined, full-step only: {overall_classical_full:.4f}")

print(f"\n{'='*60}")
print("FAIR COMPARISON")
print(f"{'='*60}")
print(f"Classical ML (full trials only):  {overall_classical_full:.4f}")
print(f"CNN-1D (full trials):              0.7392")
print(f"EEGNet (full trials):              0.7395")

        model                  paradigm  mean_balanced_acc  n_folds
random_forest        DS002680__gonogo_1           0.565346       10
random_forest       DS002680__gonogo_10           0.506323       12
random_forest       DS002680__gonogo_11           0.537755       12
random_forest       DS002680__gonogo_12           0.502447       11
random_forest       DS002680__gonogo_13           0.533476       11
random_forest        DS002680__gonogo_2           0.532747       11
random_forest        DS002680__gonogo_3           0.548801       12
random_forest        DS002680__gonogo_4           0.530708       11
random_forest        DS002680__gonogo_5           0.519995       12
random_forest        DS002680__gonogo_6           0.506412       12
random_forest        DS002680__gonogo_7           0.524463       12
random_forest        DS002680__gonogo_8           0.521038       12
random_forest        DS002680__gonogo_9           0.537000       11
random_forest DS006018__auditoryoddball         

In [83]:
# ============================================================
# Pull the real CNN-1D/EEGNet results for N2pc, N400, N170 --
# already exist on disk from last night's overnight run
# ============================================================
import pandas as pd

paradigms_to_check = ["ERP_CORE_N2pc", "ERP_CORE_N400", "ERP_CORE_N170"]

for model in ["cnn1d", "eegnet"]:
    print(f"\n=== {model} ===")
    for paradigm in paradigms_to_check:
        csv_path = project_root / "results_dl" / model / paradigm / "step_full_folds.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            print(f"{paradigm}: balanced_accuracy = {df['balanced_accuracy'].mean():.4f}")
        else:
            print(f"{paradigm}: FILE NOT FOUND at {csv_path}")


=== cnn1d ===
ERP_CORE_N2pc: balanced_accuracy = 0.4998
ERP_CORE_N400: balanced_accuracy = 0.6670
ERP_CORE_N170: balanced_accuracy = 0.6249

=== eegnet ===
ERP_CORE_N2pc: balanced_accuracy = 0.4947
ERP_CORE_N400: balanced_accuracy = 0.6918
ERP_CORE_N170: balanced_accuracy = 0.6849
